<a href="https://colab.research.google.com/github/ashok-bisht/Context-Aware_Misinformation_Detection_ML_and_Gen_AI/blob/main/notebooks/1.1-eda-data-cleaning-single-text-column.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 1. EDA, Load and Merge

* Load and read df_true and df_fake data.
* View the description of the true and fake sets.
* Label for the true and fake sets (1, 0).

## Clean the data:

* Concat the two datasets into df.
* Remove unused columns, keeping only the title, text and label.
* Remove missing rows with drop null.
* Remove extra spaces.
* Check if the number of real/fake records is equal.
* Mix the data to ensure training.
* Calculate the length of each title in a data point.
* Remove the publisher info like Reuter, CNN etc from the text from both dataset.


In [3]:
#Initialize folder paths
import os
import re
import pandas as pd

# 1. Define the base directory path on your Google Drive
base_drive_folder = "/content/drive/MyDrive/Colab Notebooks/Context-Aware Misinformation Detection"

# Define explicit input, temp and output folder paths
input_folder_path = os.path.join (base_drive_folder, "input")
temp_folder_path = os.path.join (base_drive_folder, "temp")
clean_folder_path = os.path.join (base_drive_folder, "clean")

input_file_true = os.path.join(input_folder_path, "True.csv")
input_file_fake = os.path.join(input_folder_path, "Fake.csv")
input_temp_true = os.path.join(temp_folder_path, "True_Pub_Cleaned.csv")
input_temp_fake = os.path.join(temp_folder_path, "Fake_Pub_Cleaned.csv")


## clean datasets by removing the publisher info

In [4]:

# Define the regex cleaning function
def strip_journalism_fingerprints(text, publishers_to_scrub=None):
    if not isinstance(text, str):
        return ""

    # 1. Remove the last line like 'Featured image via Al Drago-Pool/Getty Images
    # We replace it with a single period to preserve the end of the previous sentence!
    text=re.sub(r"\.\s*featured\s+image.*$", ".", text, flags=re.IGNORECASE | re.MULTILINE)
    # Remove [VIDEO], [video], [ Video ], etc., along with any surrounding spaces
    text= re.sub(r"\s*\[\s*video\s*\]\s*", " ", text, flags=re.IGNORECASE).strip()

    # Clean up spacing and weird characters/newlines at the start
    text = text.strip()
    text = text.replace("\u00a0", " ")  # Fix NBSP

    # Standardize punctuation
    punctuation_map = {
        "’": "'", "‘": "'",  # Curly apostrophes -> Straight apostrophe
        "“": '"', "”": '"',  # Curly quotes -> Straight quotes
        "–": "-", "—": "-",  # En/Em dashes -> Standard hyphen
    }
    for curly, straight in punctuation_map.items():
        text = text.replace(curly, straight)
    text = text.replace("â€™", "'")      # Fix broken UTF-8 apostrophes ("â€™" -> "'")

    # Fix "isn t" -> "isn't", "don t" -> "don't"
    text = re.sub(r"\b(isn|don|didn|doesn|can|wasn|weren|haven|hasn|hadn|won|wouldn|shouldn|couldn|aren)\s+t\b", r"\1't", text, flags=re.IGNORECASE)

    # Fix spaces around real apostrophes like "don ' t" or "don 't"
    text = re.sub(r"\b(don|didn|doesn|isn|can)\s*'\s*t\b", r"\1't", text, flags=re.IGNORECASE)

    # 2. Remove leading bracketed corrections/clarifications at the very start
    # Matches: "(In 2nd paragraph...)"
    leading_bracket_pattern = r"^\([^)]+\)\s*"
    text = re.sub(leading_bracket_pattern, "", text)

    # 3. Strip standard datelines (e.g., "NEW YORK (Reuters) - ")
    dateline_pattern = r"^[A-Z\s,]+(?:\s\([^)]+\))?\s*[\s\-\-–—]\s*"
    text = re.sub(dateline_pattern, "", text)

    # 4. Remove standalone bracketed publisher or author markers (e.g., "(Reuters)" or "By Terray Sylvester")
    # This specifically target patterns like "(Reuters)" or "By John Doe (Reuters)" left over in the text
    byline_pattern = r"(?:By\s+[A-Za-z\s]+)?\s*\([^)]+\)"
    text = re.sub(byline_pattern, "", text)

    # 5. Scrub specific publisher words anywhere else in the text
    if publishers_to_scrub:
        escaped_publishers = [re.escape(pub) for pub in publishers_to_scrub]
        scrub_pattern = r"\b(" + "|".join(escaped_publishers) + r")\b"
        text = re.sub(scrub_pattern, "XYZ", text, flags=re.IGNORECASE)

    # Final cleanup of double spaces left behind by removals
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# ============================================================
# EXECUTION PIPELINE
# ============================================================
try:

    #******************************************************
    # Process True file
    #******************************************************

    # Load the original Kaggle CSV file
    print(f"🔄 Loading dataset True from: {input_file_true}")
    df_true= pd.read_csv(input_file_true, encoding='utf-8')
    print("✂️ Stripping datelines and publisher signatures from text for True...")

    # Apply the regex function exclusively to the text column
    publishers = ["Reuters", "CNN", "Associated Press", "BBC"]
    df_true["text"] = df_true["text"].apply(lambda x: strip_journalism_fingerprints(x, publishers_to_scrub=publishers))
    # On title column
    df_true["title"] = df_true["title"].apply(lambda x: strip_journalism_fingerprints(x, publishers_to_scrub=publishers))

    # Save to the new destination file name
    print(f"💾 Saving cleaned True file to: {temp_folder_path}")
    df_true.to_csv(input_temp_true, index=False)

    #******************************************************
    # Process Fake file
    #******************************************************

    print(f"🔄 Loading dataset Fake from: {input_file_fake}")
    # Load the original Kaggle CSV file
    df_fake= pd.read_csv(input_file_fake, encoding='utf-8')

    print("✂️ Stripping datelines and publisher signatures from text for True...")

    # Apply the regex function exclusively to the text column
    df_fake["text"] = df_fake["text"].apply(lambda x: strip_journalism_fingerprints(x, publishers_to_scrub=publishers))
    # on title column
    df_fake["title"] = df_fake["title"].apply(lambda x: strip_journalism_fingerprints(x, publishers_to_scrub=publishers))

    # Save to the new destination file name
    print(f"💾 Saving cleaned Fake file to: {temp_folder_path}")
    df_fake.to_csv(input_temp_fake, index=False)

    print("✅ Process complete! The data leakage loophole has been fixed.")

except FileNotFoundError:
    print(
        f"❌ Error: Could not find 'True.csv' at '{input_folder_path}'."
        " Please make sure your Google Drive is mounted using "
        " 'from google.colab import drive; drive.mount(\"/content/drive\")'"
    )
except Exception as e:
    print(f"❌ An error occurred during processing: {str(e)}")


🔄 Loading dataset True from: /content/drive/MyDrive/Colab Notebooks/Context-Aware Misinformation Detection/input/True.csv
✂️ Stripping datelines and publisher signatures from text for True...
💾 Saving cleaned True file to: /content/drive/MyDrive/Colab Notebooks/Context-Aware Misinformation Detection/temp
🔄 Loading dataset Fake from: /content/drive/MyDrive/Colab Notebooks/Context-Aware Misinformation Detection/input/Fake.csv
✂️ Stripping datelines and publisher signatures from text for True...
💾 Saving cleaned Fake file to: /content/drive/MyDrive/Colab Notebooks/Context-Aware Misinformation Detection/temp
✅ Process complete! The data leakage loophole has been fixed.


## Merge title and text -> content and merge dataset


In [5]:
import numpy as np # linear algebra
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Add a 'label' column (0 for fake, 1 for true)
df_fake['label'] = 0
df_true['label'] = 1

# Concatenate the dataframes
df_combined = pd.concat([df_fake, df_true], ignore_index=True)

# Merge 'title' and 'text' columns into a new 'content' column
df_combined['content'] = df_combined['title'] + ". " + df_combined['text']

# Keep only 'content' and 'label' columns
df_combined = df_combined[['content', 'label']]

# Display the first few rows and info of the combined dataframe
print("Combined DataFrame Head after modifications:")
display(df_combined.head())
print("\nCombined DataFrame Info after modifications:")
df_combined.info()

Combined DataFrame Head after modifications:


,content,label
0,Donald Trump Sends Out Embarrassing New Year's...,0
1,Drunk Bragging Trump Staffer Started Russian C...,0
2,Sheriff David Clarke Becomes An Internet Joke ...,0
3,Trump Is So Obsessed He Even Has Obama's Name ...,0
4,Pope Francis Just Called Out Donald Trump Duri...,0



Combined DataFrame Info after modifications:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44898 entries, 0 to 44897
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   content  44898 non-null  object
 1   label    44898 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 701.7+ KB


In [6]:
import pandas as pd
# Check label distribution
print ("Label distribution:")
print(df_combined['label'].value_counts())
print ("--"*50)
print ("Combined DataFrame Shape:")
print (df_combined.shape)

Label distribution:
label
0    23481
1    21417
Name: count, dtype: int64
----------------------------------------------------------------------------------------------------
Combined DataFrame Shape:
(44898, 2)


In [7]:
# Columns 'date' and 'subject' were removed in a previous step by selecting only 'content' and 'label'
print ("DataFrame shape:")
print (df_combined.shape)

DataFrame shape:
(44898, 2)


In [9]:
# Drop NA records
df_combined.dropna(inplace=True)
print ("After Drop NA", df_combined.shape)

# Trim spaces in 'content'
df_combined['content'] = df_combined['content'].astype(str).str.strip()
print ("After Trim Space", df_combined.shape)

# Remove duplicate records
df_combined.drop_duplicates(inplace=True)
print ("After Duplicate removal", df_combined.shape)

After Drop NA (39099, 2)
After Trim Space (39099, 2)
After Duplicate removal (39099, 2)


In [10]:
#check true vs fake
print ("Check the distribution of True and False")
print(df_combined["label"].value_counts())

Check the distribution of True and False
label
1    21195
0    17904
Name: count, dtype: int64


In [11]:
# Randomly mix the data
df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)

In [12]:
df_combined.head()

,content,label
0,This Cop Sees Black Lives Matter In A Way That...,0
1,Upgrade with a dab of TPP may be U.S. recipe f...,1
2,SURRENDER: Former Trump Campaign Manager Charg...,0
3,"Sorry Conservatives, Obamacare Isn't Killing J...",0
4,Venezuela's opposition takes EU human rights p...,1


## Save Cleaned Dataset to the google drive Path


In [13]:
# Save the df_combined to this location
import os

# 1. Define the Google Drive folder and file name
combined_csv_path = os.path.join(clean_folder_path, 'combined_news_single_column.csv')

# 2. Ensure the directory exists
os.makedirs(clean_folder_path, exist_ok=True)

print("Saving DataFrame to Google Drive... Please wait.")

# 3. Save to CSV (index=False prevents pandas from adding an extra row numbers column)
df_combined.to_csv(combined_csv_path, index=False)

print(f"🎉 Success! Dataset successfully saved to: {combined_csv_path}")



Saving DataFrame to Google Drive... Please wait.
🎉 Success! Dataset successfully saved to: /content/drive/MyDrive/Colab Notebooks/Context-Aware Misinformation Detection/clean/combined_news_single_column.csv


In [14]:
# Install spaCy and download the English model
!pip install spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 97.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


### 2. Text Cleaning with spaCy

Now, clean the text data using spaCy. This involves:
-   **Loading the spaCy model**: `en_core_web_sm`.
-   **Tokenization**: Breaking down text into individual words.
-   **Stopword Removal**: Removing common words that don't add much meaning.
-   **Lowercasing**: Converting all text to lowercase.
-   **Removing Punctuation and Special Characters**.
-   **Lemmatization**: Reducing words to their base form.

After cleaning, save the processed DataFrame to a new CSV file named `cleaned_news_data.csv`.

In [ ]:
# run if the colab fails in middle.
#load the DF from the drive
# import pandas as pd
# #pd.set_option('display.max_colwidth', 50)
# df_combined = pd.read_csv(combined_csv_path)
# df_combined.head()

In [15]:
import os
from datetime import datetime
import pandas as pd
import spacy

# ==========================================
# ⚙️ CONFIGURATION BLOCK
# ==========================================
START_ROW = 0
N = 1000
# ==========================================

spacy_csv_file = os.path.join(clean_folder_path, 'cleaned_news_spacy.csv')
if os.path.exists(spacy_csv_file):
    os.remove(spacy_csv_file)

nlp = spacy.load('en_core_web_sm', disable=['tok2vec', 'parser', 'ner'])

def clean_text_list(text_list):
    cleaned = []
    for doc in nlp.pipe(text_list, batch_size=250, n_process=-1):
        tokens = [token.lemma_ for token in doc if token.is_alpha and not token.is_stop and not token.pos_ == "PROPN"]
        cleaned.append(" ".join(tokens))
    return cleaned

total_rows = len(df_combined)
for start in range(START_ROW, total_rows, N):
    end = min(start + N, total_rows)
    batch_df = df_combined.iloc[start:end].copy()

    # Only process 'content' as 'title' is no longer separate
    content_texts = batch_df['content'].fillna("").astype(str).str.lower().tolist()
    batch_df['cleaned_content'] = clean_text_list(content_texts)

    if start == 0:
        batch_df[['content', 'label', 'cleaned_content']].to_csv(spacy_csv_file, index=False, mode='w')
    else:
        batch_df[['content', 'label', 'cleaned_content']].to_csv(spacy_csv_file, index=False, mode='a', header=False)

    print(f"✅ Processed rows {start} to {end-1}")

print(f"Finished processing. Saved to: {spacy_csv_file}")

✅ Processed rows 0 to 999
✅ Processed rows 1000 to 1999
✅ Processed rows 2000 to 2999
✅ Processed rows 3000 to 3999
✅ Processed rows 4000 to 4999
✅ Processed rows 5000 to 5999
✅ Processed rows 6000 to 6999
✅ Processed rows 7000 to 7999
✅ Processed rows 8000 to 8999
✅ Processed rows 9000 to 9999
✅ Processed rows 10000 to 10999
✅ Processed rows 11000 to 11999
✅ Processed rows 12000 to 12999
✅ Processed rows 13000 to 13999
✅ Processed rows 14000 to 14999
✅ Processed rows 15000 to 15999
✅ Processed rows 16000 to 16999
✅ Processed rows 17000 to 17999
✅ Processed rows 18000 to 18999
✅ Processed rows 19000 to 19999
✅ Processed rows 20000 to 20999
✅ Processed rows 21000 to 21999
✅ Processed rows 22000 to 22999
✅ Processed rows 23000 to 23999
✅ Processed rows 24000 to 24999
✅ Processed rows 25000 to 25999
✅ Processed rows 26000 to 26999
✅ Processed rows 27000 to 27999
✅ Processed rows 28000 to 28999
✅ Processed rows 29000 to 29999
✅ Processed rows 30000 to 30999
✅ Processed rows 31000 to 31999


### 3. TF-IDF Vectorization

Finally, perform TF-IDF (Term Frequency-Inverse Document Frequency) vectorization on the `cleaned_content` column. TF-IDF is a numerical statistic that reflects how important a word is to a document in a collection or corpus.

I will use `TfidfVectorizer` from `sklearn.feature_extraction.text` to convert the text data into a matrix of TF-IDF features. The resulting TF-IDF matrix will be saved as a new CSV file named `tfidf_vectors.csv` for use by other team members.

In [ ]:
import os
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse
import numpy as np

df_spacy = pd.read_csv(spacy_csv_file)
df_spacy['cleaned_content'] = df_spacy['cleaned_content'].fillna("")

# Updated ColumnTransformer to only use 'cleaned_content'
preprocessor = ColumnTransformer(
    transformers=[
        ('content_tfidf', TfidfVectorizer(max_features=25000), 'cleaned_content')
    ],
    remainder='passthrough'
)

print("Vectorizing features...")
tfidf_matrix = preprocessor.fit_transform(df_spacy)
feature_names = preprocessor.get_feature_names_out()

sparse_matrix_file_path = os.path.join(base_drive_folder, 'tfidf_sparse_matrix.npz')
feature_names_file_path = os.path.join(base_drive_folder, 'tfidf_feature_names.npy')
labels_file_path = os.path.join(base_drive_folder, 'tfidf_labels.csv')

scipy.sparse.save_npz(sparse_matrix_file_path, tfidf_matrix)
np.save(feature_names_file_path, feature_names)
df_spacy['label'].to_csv(labels_file_path, index=False)

print(f"Files saved to {base_drive_folder}")

Vectorizing features in parallel...

Sparse TF-IDF matrix successfully saved to: /content/drive/MyDrive/Colab Notebooks/Context-Aware Misinformation Detection/tfidf_sparse_matrix.npz
TF-IDF feature names successfully saved to: /content/drive/MyDrive/Colab Notebooks/Context-Aware Misinformation Detection/tfidf_feature_names.npy
Classification labels successfully saved to: /content/drive/MyDrive/Colab Notebooks/Context-Aware Misinformation Detection/tfidf_labels.csv
